In [1]:
import pandas as pd
import pyodbc

In [2]:
conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 18 for SQL Server};"
    r"SERVER=.\SQLEXPRESS;"
    r"DATABASE=ShopifyMarketplace;"
    r"Trusted_Connection=yes;"
    r"TrustServerCertificate=yes;"
)

print("Connected successfully!")

Connected successfully!


In [3]:
# Question 10: Load 1-star review text into Python for complaint analysis
# 问题 10：将 1-star Review 文本读取到 Python，用于用户投诉分析

query = """
SELECT
    app_id,
    body
FROM reviews
WHERE rating = 1
  AND has_review_text = 1;
"""

one_star_reviews = pd.read_sql(query, conn)

print(one_star_reviews.shape)
one_star_reviews.head()

C:\Users\xiongsongsong\AppData\Local\Temp\ipykernel_45588\2441066749.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  one_star_reviews = pd.read_sql(query, conn)


(32938, 2)


,app_id,body
0,86aa48c8-feeb-45bb-a3c0-e830884a360e,"It is a very costly solution, which unfairly l..."
1,e52682e1-5fb5-4a47-977e-718e5a1f9beb,"5 months of trying to sync 2 stores, 3dcart an..."
2,dcc2a9d5-d355-42d8-9dd9-01e85fc36a40,They are scammers dont work with them!!! I hav...
3,5bc7a3fc-cbba-4c58-b6fe-1adefc7c1f53,They asked me on day ONE to leave a 5 star rev...
4,00aecf4c-ad80-4bcf-a4d7-9fa9ddcc62b0,tres mal expliqué je comprend tres mal


In [4]:
# Step 1: Clean 1-star review text
# 第一步：清洗 1-star Review 文本

import re

def clean_review_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

one_star_reviews["clean_body"] = one_star_reviews["body"].apply(clean_review_text)

one_star_reviews[["body", "clean_body"]].head()

,body,clean_body
0,"It is a very costly solution, which unfairly l...",it is a very costly solution which unfairly lo...
1,"5 months of trying to sync 2 stores, 3dcart an...",months of trying to sync stores dcart and shop...
2,They are scammers dont work with them!!! I hav...,they are scammers dont work with them i have d...
3,They asked me on day ONE to leave a 5 star rev...,they asked me on day one to leave a star revie...
4,tres mal expliqué je comprend tres mal,tres mal expliqu je comprend tres mal


In [5]:
# Step 2: Remove common stopwords
# 第二步：删除常见无意义词

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)

def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

one_star_reviews["clean_body_no_stopwords"] = (
    one_star_reviews["clean_body"].apply(remove_stopwords)
)

one_star_reviews[
    ["clean_body", "clean_body_no_stopwords"]
].head()

,clean_body,clean_body_no_stopwords
0,it is a very costly solution which unfairly lo...,costly solution unfairly locks contract automa...
1,months of trying to sync stores dcart and shop...,months trying sync stores dcart shopify gave r...
2,they are scammers dont work with them i have d...,scammers dont work downloaded app weeks ago si...
3,they asked me on day one to leave a star revie...,asked day leave star review large discount pai...
4,tres mal expliqu je comprend tres mal,tres mal expliqu je comprend tres mal


In [6]:
# Step 3: Find the most common words in 1-star reviews
# 第三步：统计 1-star Reviews 中最常见的关键词

from collections import Counter

all_words = " ".join(
    one_star_reviews["clean_body_no_stopwords"]
).split()

word_counts = Counter(all_words)

top_words = word_counts.most_common(30)

top_words

[('app', 34715),
 ('t', 21146),
 ('support', 12078),
 ('s', 9696),
 ('shopify', 8933),
 ('customer', 8630),
 ('time', 7812),
 ('work', 7383),
 ('service', 6773),
 ('just', 6630),
 ('use', 6019),
 ('don', 5974),
 ('products', 5515),
 ('store', 5389),
 ('product', 4879),
 ('customers', 4565),
 ('help', 4203),
 ('days', 4151),
 ('like', 4117),
 ('issue', 3871),
 ('free', 3864),
 ('does', 3822),
 ('email', 3795),
 ('order', 3679),
 ('doesn', 3655),
 ('working', 3558),
 ('review', 3528),
 ('did', 3392),
 ('money', 3368),
 ('ve', 3321)]

In [7]:
# Step 4: Improve text cleaning and preserve negation
# 第四步：改进文本清洗，并保留否定含义

import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from collections import Counter

def clean_review_text_v2(text):
    text = text.lower()

    # Normalize common contractions
    # 统一常见英文缩写
    text = re.sub(r"\bcan't\b", "cannot", text)
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bdon't\b", "do not", text)
    text = re.sub(r"\bdoesn't\b", "does not", text)
    text = re.sub(r"\bdidn't\b", "did not", text)
    text = re.sub(r"\bisn't\b", "is not", text)
    text = re.sub(r"\bwasn't\b", "was not", text)
    text = re.sub(r"\bweren't\b", "were not", text)
    text = re.sub(r"\bcouldn't\b", "could not", text)
    text = re.sub(r"\bwouldn't\b", "would not", text)
    text = re.sub(r"\bshouldn't\b", "should not", text)

    # Remove URLs
    # 删除网址
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Keep English letters only
    # 只保留英文字母
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces
    # 删除多余空格
    text = re.sub(r"\s+", " ", text).strip()

    return text


stop_words = set(ENGLISH_STOP_WORDS)

# Keep negation words because they are important in complaints
# 保留否定词，因为它们对投诉分析很重要
stop_words.discard("not")
stop_words.discard("no")
stop_words.discard("nor")

# Remove generic marketplace words
# 删除过于通用、分析价值较低的词
custom_stopwords = {
    "app",
    "shopify",
    "store",
    "product",
    "products",
    "review",
    "reviews"
}

stop_words.update(custom_stopwords)


def remove_stopwords_v2(text):
    words = text.split()
    return " ".join(
        word for word in words
        if word not in stop_words
    )


one_star_reviews["clean_body_v2"] = (
    one_star_reviews["body"]
    .apply(clean_review_text_v2)
    .apply(remove_stopwords_v2)
)


all_words = " ".join(
    one_star_reviews["clean_body_v2"]
).split()

word_counts = Counter(all_words)

top_words = word_counts.most_common(30)

top_words

[('not', 44360),
 ('no', 12842),
 ('support', 12078),
 ('s', 9696),
 ('customer', 8630),
 ('time', 7812),
 ('work', 7383),
 ('does', 7284),
 ('service', 6773),
 ('just', 6630),
 ('use', 6019),
 ('did', 5622),
 ('customers', 4565),
 ('help', 4203),
 ('days', 4151),
 ('like', 4117),
 ('issue', 3871),
 ('free', 3864),
 ('email', 3795),
 ('order', 3679),
 ('working', 3558),
 ('money', 3368),
 ('ve', 3321),
 ('m', 3289),
 ('account', 3251),
 ('t', 3192),
 ('business', 3182),
 ('issues', 3163),
 ('orders', 3104),
 ('using', 3100)]

In [8]:
# Step 5: Final text cleaning
# 第五步：最终清洗文本

extra_stopwords = {
    "s", "t", "ve", "m", "d", "ll", "re",
    "just", "did", "does"
}

final_stop_words = stop_words.union(extra_stopwords)

def final_clean(text):
    words = text.split()
    words = [
        word for word in words
        if word not in final_stop_words
        and len(word) > 1
    ]
    return " ".join(words)

one_star_reviews["final_clean_body"] = (
    one_star_reviews["clean_body_v2"].apply(final_clean)
)

all_words = " ".join(
    one_star_reviews["final_clean_body"]
).split()

final_word_counts = Counter(all_words)

final_word_counts.most_common(30)

[('not', 44360),
 ('no', 12842),
 ('support', 12078),
 ('customer', 8630),
 ('time', 7812),
 ('work', 7383),
 ('service', 6773),
 ('use', 6019),
 ('customers', 4565),
 ('help', 4203),
 ('days', 4151),
 ('like', 4117),
 ('issue', 3871),
 ('free', 3864),
 ('email', 3795),
 ('order', 3679),
 ('working', 3558),
 ('money', 3368),
 ('account', 3251),
 ('business', 3182),
 ('issues', 3163),
 ('orders', 3104),
 ('using', 3100),
 ('page', 3052),
 ('month', 2890),
 ('good', 2841),
 ('want', 2799),
 ('company', 2766),
 ('need', 2742),
 ('make', 2664)]

In [9]:
# Step 6: Find the most common two-word phrases in 1-star reviews
# 第六步：统计 1-star Reviews 中最常见的双词短语

from collections import Counter

bigram_counts = Counter()

for text in one_star_reviews["final_clean_body"]:
    words = text.split()

    bigrams = zip(words, words[1:])

    bigram_counts.update(bigrams)

top_bigrams = bigram_counts.most_common(30)

top_bigrams

[(('not', 'work'), 3883),
 (('customer', 'service'), 3873),
 (('customer', 'support'), 1535),
 (('not', 'working'), 1492),
 (('not', 'recommend'), 1354),
 (('waste', 'time'), 1322),
 (('not', 'use'), 1182),
 (('support', 'team'), 927),
 (('support', 'not'), 901),
 (('no', 'response'), 839),
 (('not', 'know'), 799),
 (('not', 'able'), 714),
 (('no', 'way'), 704),
 (('stay', 'away'), 607),
 (('free', 'trial'), 593),
 (('free', 'plan'), 575),
 (('not', 'good'), 569),
 (('no', 'longer'), 565),
 (('not', 'worth'), 558),
 (('service', 'not'), 546),
 (('not', 'want'), 513),
 (('not', 'install'), 510),
 (('not', 'waste'), 508),
 (('not', 'sure'), 474),
 (('no', 'support'), 457),
 (('contacted', 'support'), 453),
 (('work', 'not'), 439),
 (('not', 'help'), 382),
 (('no', 'help'), 379),
 (('not', 'buy'), 379)]

In [10]:
# Step 7: Classify 1-star reviews into complaint themes
# 第七步：将 1-star Reviews 分类为主要投诉主题

complaint_themes = {
    "Technical issues": [
        "not work", "not working", "does not work", "cannot use",
        "not install", "error", "bug", "broken"
    ],
    "Customer support": [
        "customer support", "customer service", "support team",
        "no support", "no response", "contacted support"
    ],
    "Pricing / billing": [
        "money", "charge", "charged", "billing",
        "refund", "expensive", "price", "payment"
    ],
    "Time / effort": [
        "waste time", "waste of time", "time consuming"
    ],
    "Negative recommendation": [
        "not recommend", "stay away", "do not recommend"
    ]
}

def classify_themes(text):
    matched = []

    for theme, keywords in complaint_themes.items():
        for keyword in keywords:
            if keyword in text:
                matched.append(theme)
                break

    return matched

one_star_reviews["themes"] = (
    one_star_reviews["final_clean_body"].apply(classify_themes)
)

one_star_reviews[["final_clean_body", "themes"]].head(10)

,final_clean_body,themes
0,costly solution unfairly locks contract automa...,[]
1,months trying sync stores dcart gave not refun...,[Pricing / billing]
2,scammers dont work downloaded weeks ago site g...,[Customer support]
3,asked day leave star large discount paid plan ...,[Pricing / billing]
4,tres mal expliqu je comprend tres mal,[]
5,poor customer support accidental subscription ...,"[Customer support, Pricing / billing]"
6,,[]
7,unfortunately mid campaign went received zero ...,[]
8,sucks lot data way upload template download en...,"[Technical issues, Customer support]"
9,downloaded add multiple languages shop user un...,[]


In [11]:
# Step 8: Count complaint themes
# 第八步：统计各投诉主题出现次数和占比

from collections import Counter

theme_counts = Counter()

for themes in one_star_reviews["themes"]:
    theme_counts.update(themes)

theme_summary = pd.DataFrame(
    theme_counts.items(),
    columns=["theme", "review_count"]
)

theme_summary["percentage"] = (
    theme_summary["review_count"]
    / len(one_star_reviews)
    * 100
).round(2)

theme_summary = theme_summary.sort_values(
    "review_count",
    ascending=False
)

theme_summary

,theme,review_count,percentage
0,Pricing / billing,7962,24.17
2,Technical issues,7671,23.29
1,Customer support,6173,18.74
3,Negative recommendation,2066,6.27
4,Time / effort,1397,4.24


In [12]:
# Step 9: Check complaint-theme classification coverage
# 第九步：检查投诉主题分类的覆盖率

matched_reviews = one_star_reviews["themes"].apply(len).gt(0).sum()
unmatched_reviews = one_star_reviews["themes"].apply(len).eq(0).sum()

coverage_summary = pd.DataFrame({
    "type": ["Matched", "Unmatched"],
    "review_count": [matched_reviews, unmatched_reviews]
})

coverage_summary["percentage"] = (
    coverage_summary["review_count"]
    / len(one_star_reviews)
    * 100
).round(2)

coverage_summary

,type,review_count,percentage
0,Matched,18197,55.25
1,Unmatched,14741,44.75


In [13]:
# Step 10: Analyze unmatched 1-star reviews
# 第十步：分析尚未匹配投诉主题的 1-star Reviews

unmatched_reviews = one_star_reviews[
    one_star_reviews["themes"].apply(len) == 0
].copy()

print("Unmatched reviews:", len(unmatched_reviews))

unmatched_bigram_counts = Counter()

for text in unmatched_reviews["final_clean_body"]:
    words = text.split()
    bigrams = zip(words, words[1:])
    unmatched_bigram_counts.update(bigrams)

unmatched_bigram_counts.most_common(30)

Unmatched reviews: 14741


[(('not', 'use'), 345),
 (('no', 'way'), 274),
 (('free', 'plan'), 259),
 (('not', 'know'), 253),
 (('support', 'not'), 248),
 (('not', 'able'), 238),
 (('not', 'good'), 225),
 (('not', 'sure'), 189),
 (('no', 'longer'), 188),
 (('free', 'trial'), 181),
 (('not', 'worth'), 166),
 (('not', 'allow'), 149),
 (('stopped', 'working'), 147),
 (('not', 'free'), 134),
 (('not', 'want'), 131),
 (('user', 'friendly'), 130),
 (('google', 'shopping'), 127),
 (('not', 'let'), 123),
 (('no', 'help'), 120),
 (('merchant', 'center'), 120),
 (('not', 'help'), 119),
 (('doesnt', 'work'), 119),
 (('customers', 'not'), 117),
 (('free', 'version'), 113),
 (('looks', 'like'), 111),
 (('no', 'reply'), 107),
 (('use', 'not'), 107),
 (('google', 'merchant'), 107),
 (('not', 'support'), 103),
 (('not', 'like'), 98)]

In [14]:
# Step 11: Improve complaint-theme classification
# 第十一步：改进投诉主题分类规则

complaint_themes_v2 = {
    "Technical issues": [
        "not work",
        "not working",
        "does not work",
        "doesnt work",
        "stopped working",
        "cannot use",
        "not use",
        "not able",
        "not install",
        "error",
        "bug",
        "broken"
    ],

    "Customer support": [
        "customer support",
        "customer service",
        "support team",
        "no support",
        "no response",
        "no reply",
        "contacted support",
        "support not",
        "not support"
    ],

    "Pricing / billing": [
        "money",
        "charge",
        "charged",
        "billing",
        "refund",
        "expensive",
        "price",
        "payment",
        "free plan",
        "free trial",
        "free version",
        "not free"
    ],

    "Usability": [
        "user friendly",
        "not user friendly",
        "hard to use",
        "difficult to use",
        "not easy"
    ],

    "Time / effort": [
        "waste time",
        "waste of time",
        "time consuming"
    ],

    "Negative recommendation": [
        "not recommend",
        "stay away",
        "do not recommend"
    ]
}


def classify_themes_v2(text):
    matched = []

    for theme, keywords in complaint_themes_v2.items():
        for keyword in keywords:
            if keyword in text:
                matched.append(theme)
                break

    return matched


one_star_reviews["themes_v2"] = (
    one_star_reviews["final_clean_body"].apply(classify_themes_v2)
)